# Slice-aware RAN training scenarios

The paper-style six-UE layout is now slice-aware: each side contains one eMBB, one URLLC, and one mMTC UE, all initially served by the center cell.

Across the catalog:

- **circles** are overlap UEs that may become A3 handover candidates;
- **squares** are fixed-core UEs used as persistent background/target load;
- blue = eMBB, red = URLLC, green = mMTC.

In [ ]:
import math, os, sys, textwrap
from pathlib import Path
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-cache")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Circle
import numpy as np

ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))

from global_ppo_3gnb_env import GlobalPPO3GNBEnv
from upper_agent_training_scenarios import CENTER_LEFT_RIGHT_GNB_CONFIGS, UPPER_TRAINING_SCENARIOS

SLICE_COLORS = {"eMBB": "#4C78A8", "URLLC": "#E15759", "mMTC": "#59A14F"}
PURPOSE = {
 "paper_six_ue_slice_aware": "Basic slice-aware proof: test independent eMBB, URLLC and mMTC release from the same center cell.",
 "fixed_center_embb_left_right": "Single-slice baseline: isolate eMBB bias, quota and left/right admission behavior.",
 "mixed_slices_center_overlap": "Dense mixed overlap: verify that all three slice actions and quotas operate simultaneously.",
 "embb_overlap_preloaded_targets": "Safe admission test: target cells already carry fixed eMBB load.",
 "asymmetric_embb_target_loads": "Directional choice test: prefer the lighter target when left/right fixed loads differ.",
 "urllc_mmtc_overlap_fixed_embb": "Cross-slice coexistence: balance URLLC/mMTC while eMBB remains fixed background load.",
 "mixed_overlap_with_fixed_slice_loads": "Hard mixed case: overlap slices interact with fixed loads from different slices/cells.",
}
SEED = 123

In [ ]:
def make_env(name):
    return GlobalPPO3GNBEnv(seed=SEED, gnb_configs=CENTER_LEFT_RIGHT_GNB_CONFIGS,
        scenario_mode="curriculum", training_scenarios=name, scenario_selection="cycle",
        upper_window_seconds=1.0, local_steps_per_global=10, radio_substeps=2,
        terminal_reward_only=False, max_handovers_per_local_step=3,
        a3_handover_cooldown_s=2.0, a3_min_residence_s=2.0)

def balancing_action(loads):
    loads=np.asarray(loads,float); means=loads.mean(axis=0,keepdims=True); a=np.zeros_like(loads)
    a[loads>means+0.05]=-0.8; a[loads<means-0.05]=0.5
    return a.astype(np.float32).reshape(-1)

records=[]
for scenario in UPPER_TRAINING_SCENARIOS:
    env=make_env(scenario.name); _obs,reset=env.reset(seed=SEED); ues=list(env.base_env.get_all_ues())
    rows=[]; fixed_ids=set(); cursor=0
    for group in scenario.groups:
        for ue in ues[cursor:cursor+group.count]:
            coverage=tuple(math.hypot(ue.x-g.x,ue.y-g.y)<=g.coverage_radius for g in env.base_env.gnbs)
            rows.append(dict(id=int(ue.id),x=float(ue.x),y=float(ue.y),slice=group.slice_type,
                source=group.source_gnb,region=group.placement_region,coverage=coverage))
            if group.placement_region=="fixed_core": fixed_ids.add(int(ue.id))
        cursor+=group.count
    before=np.asarray(reset["load_matrix"],float)
    _obs,reward,_t,_tr,info=env.step(balancing_action(before)); events=list(env.base_env.handover_events)
    assert not fixed_ids.intersection(e["ue_id"] for e in events)
    records.append(dict(scenario=scenario,ues=rows,before=before,after=np.asarray(info["load_matrix"],float),
        events=events,reward=float(reward)))
    env.close()

print(f"{'scenario':>38} {'UE':>3} {'overlap':>7} {'fixed':>5} {'HO':>3}  purpose")
for r in records:
    ov=sum(u['region']=='overlap' for u in r['ues']); fx=len(r['ues'])-ov
    print(f"{r['scenario'].name:>38} {len(r['ues']):3d} {ov:7d} {fx:5d} {len(r['events']):3d}  {PURPOSE[r['scenario'].name]}")

## Paper-style six-UE slice-aware scenario

This is the direct slice-aware version of the requested figure. The left and right clusters have identical slice composition, allowing the upper agent to request different release volumes per slice.

In [ ]:
paper=next(r for r in records if r['scenario'].name=='paper_six_ue_slice_aware')
fig,(ax,heat)=plt.subplots(1,2,figsize=(14,5),gridspec_kw={'width_ratios':[2.1,1]})
for cfg in CENTER_LEFT_RIGHT_GNB_CONFIGS:
    ax.add_patch(Circle((cfg['x'],cfg['y']),cfg['coverage_radius'],fill=False,ls='--',lw=1.3,alpha=.28,color='#3498db'))
    ax.scatter(cfg['x'],cfg['y'],marker='^',s=180,color='#34495e'); ax.text(cfg['x'],45,f"gNB{cfg['id']}",ha='center',weight='bold')
for u in paper['ues']:
    ax.scatter(u['x'],u['y'],s=95,color=SLICE_COLORS[u['slice']],edgecolor='black',zorder=5)
    ax.text(u['x'],u['y']+20,f"UE{u['id']}\n{u['slice']}",ha='center',fontsize=8)
ax.set(xlim=(-850,850),ylim=(-530,530),aspect='equal',xlabel='x (m)',ylabel='y (m)',title='Six fixed center-served UEs: one UE per slice on each side')
im=heat.imshow(paper['before'],vmin=0,vmax=1,cmap='YlOrRd'); heat.set_xticks(range(3),['eMBB','URLLC','mMTC']); heat.set_yticks(range(3),['left','center','right'])
heat.set_title('Initial slice load')
for i in range(3):
 for j in range(3): heat.text(j,i,f"{paper['before'][i,j]:.2f}",ha='center',va='center')
plt.tight_layout(); plt.show()
assert {u['slice'] for u in paper['ues']}=={'eMBB','URLLC','mMTC'}
assert all(sum(u['coverage'])==3 for u in paper['ues'])

## Scenario cards: geometry plus training intention

Each card explains what PPO should learn from that case. This makes the catalog a training design document, not only a topology plot.

In [ ]:
fig,axes=plt.subplots(4,2,figsize=(15,18)); axes=axes.ravel()
for ax,r in zip(axes,records):
 for cfg in CENTER_LEFT_RIGHT_GNB_CONFIGS:
  ax.add_patch(Circle((cfg['x'],0),cfg['coverage_radius'],fill=False,ls='--',alpha=.18,color='#555'))
  ax.scatter(cfg['x'],0,marker='^',s=90,color='#333')
 for u in r['ues']:
  marker='o' if u['region']=='overlap' else 's'
  ax.scatter(u['x'],u['y'],marker=marker,s=58,color=SLICE_COLORS[u['slice']],edgecolor='black',lw=.7)
 ax.set(xlim=(-900,900),ylim=(-550,550),aspect='equal'); ax.set_xticks([]); ax.set_yticks([])
 ax.set_title(r['scenario'].name,weight='bold')
 ax.text(.5,-.06,textwrap.fill(PURPOSE[r['scenario'].name],68),transform=ax.transAxes,ha='center',va='top',fontsize=9)
for ax in axes[len(records):]: ax.axis('off')
legend=[Line2D([0],[0],marker='o',color='w',label=s,markerfacecolor=c,markeredgecolor='black',markersize=9) for s,c in SLICE_COLORS.items()]
legend += [Line2D([0],[0],marker='o',color='w',label='overlap / movable',markerfacecolor='white',markeredgecolor='black',markersize=9),Line2D([0],[0],marker='s',color='w',label='fixed-core load',markerfacecolor='white',markeredgecolor='black',markersize=9)]
fig.legend(handles=legend,loc='lower center',ncol=5); plt.tight_layout(rect=(0,.04,1,1)); plt.show()

## Load and handover response

Rows are cells `[left, center, right]`; columns are slices `[eMBB, URLLC, mMTC]`. Fixed-core UE IDs are checked against handover events and must never move.

In [ ]:
fig,axes=plt.subplots(len(records),2,figsize=(9,3*len(records)))
for row,r in enumerate(records):
 for col,(matrix,label) in enumerate(((r['before'],'before'),(r['after'],'after'))):
  ax=axes[row,col]; ax.imshow(matrix,vmin=0,vmax=1,cmap='YlOrRd'); ax.set_xticks(range(3),['eMBB','URLLC','mMTC']); ax.set_yticks(range(3),['L','C','R'])
  ax.set_title(f"{r['scenario'].name} — {label}",fontsize=9)
  for i in range(3):
   for j in range(3): ax.text(j,i,f"{matrix[i,j]:.2f}",ha='center',va='center',fontsize=8)
plt.tight_layout(); plt.show()

assert len(records)==7
assert any(u['region']=='fixed_core' for r in records for u in r['ues'])
for r in records:
 for u in r['ues']:
  assert sum(u['coverage'])==(3 if u['region']=='overlap' else 1)
print('PASS: paper topology is slice-aware; every scenario has an explicit training purpose; fixed-core loads never hand over.')